In [ ]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
import numpy as np

## Datasets

In [ ]:
YELLOW_TAXI_TRIP_RECORDS_URL_JAN = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
YELLOW_TAXI_TRIP_RECORDS_URL_FEB = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"

Question 1: Read the data for January. How many columns are there?

In [ ]:
jan_df = pd.read_parquet(YELLOW_TAXI_TRIP_RECORDS_URL_JAN)
jan_df.head()

In [ ]:
jan_df.shape[1]

There are **19** columns

Question 2: Computing duration. What's the standard deviation of the trips duration in January?

In [ ]:
jan_df["tpep_pickup_datetime"] = pd.to_datetime(jan_df["tpep_pickup_datetime"])
jan_df["tpep_dropoff_datetime"] = pd.to_datetime(jan_df["tpep_dropoff_datetime"])
jan_df.loc[:, "duration"] = (jan_df["tpep_dropoff_datetime"] - jan_df["tpep_pickup_datetime"]).dt.total_seconds() / 60.0
jan_df["duration"].std()

Next, we need to check the distribution of the duration variable. There are some outliers. Let's remove them and keep only the records where the duration was between 1 and 60 minutes (inclusive).

What fraction of the records left after you dropped the outliers?



In [ ]:
jan_good_duration_df = jan_df[(jan_df["duration"] >=1) & (jan_df["duration"] <= 60)]
jan_good_duration_df

In [ ]:
jan_good_duration_df.shape[0] / jan_df.shape[0] * 100

Let's apply one-hot encoding to the pickup and dropoff location IDs. We'll use only these two features for our model.

- Turn the dataframe into a list of dictionaries (remember to re-cast the ids to strings - otherwise it will label encode them)
- Fit a dictionary vectorizer
- Get a feature matrix from it

What's the dimensionality of this matrix (number of columns)?



In [ ]:
categorical_features = [
    "PULocationID",
    "DOLocationID"
]
jan_good_duration_df.loc[:, categorical_features] = jan_good_duration_df[categorical_features].astype(str)
train_dict = jan_good_duration_df[categorical_features].to_dict(orient="records")

dv = DictVectorizer()
X_train = dv.fit_transform(train_dict)
X_train

In [ ]:
y_train = jan_good_duration_df["duration"].values

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

Question:Now let's use the feature matrix from the previous step to train a model.

Train a plain linear regression model with default parameters, where duration is the response variable.
Calculate the RMSE of the model on the training data

What's the RMSE on train?

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)

In [ ]:
rmse = root_mean_squared_error(y_true=y_train, y_pred=y_train_pred)
rmse

In [ ]:
def read_and_process_dataset(file: str) -> pd.DataFrame:
    df = pd.read_parquet(file)
    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])
    df.loc[:, "duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60.0
    df = df[(df["duration"] >=1) & (df["duration"] <= 60)]
    categorical_features = [
        "PULocationID",
        "DOLocationID"
    ]
    df.loc[:, categorical_features] = df[categorical_features].astype(str)
    return df

In [ ]:
df_train = read_and_process_dataset(file=YELLOW_TAXI_TRIP_RECORDS_URL_JAN)
df_val = read_and_process_dataset(file=YELLOW_TAXI_TRIP_RECORDS_URL_FEB)

In [ ]:
train_dict = df_train[categorical_features].to_dict(orient="records")
val_dict = df_val[categorical_features].to_dict(orient="records")

dv = DictVectorizer()

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

y_train = df_train["duration"].values
y_val = df_val["duration"].values

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)
y_val_pred = lr.predict(X_val)

In [ ]:
train_rmse = root_mean_squared_error(y_true=y_train, y_pred=y_train_pred)
val_rmse = root_mean_squared_error(y_true=y_val, y_pred=y_val_pred)